# Notebook 5: The `sbi` Package

In Notebook 4 we built NPE and NLE by hand. Now we solve the same problem with [`sbi`](https://sbi.readthedocs.io/en/stable/), the most widely used Python package for simulation-based inference. It provides well-tested flows, the training loop, samplers and diagnostics, so that a complete analysis takes a handful of lines.

This notebook is both a continuation of the tutorial and a quickstart for the package.

**What you will learn**
- the `sbi` workflow: prior, simulations, `NPE` / `NLE`, posterior
- what amortized inference means
- how to choose and configure the density estimator and the MCMC sampler
- how the number of simulations affects the result
- how to validate posteriors with the built-in diagnostics (TARP, SBC), and how to read the result
- how to save a posterior, and where to find more

**Runtime:** about two minutes on a laptop.

*Version note:* this notebook was written for `sbi` 0.27. Older tutorials use the names `SNPE` and `SNLE`, which are deprecated in 0.27 and will be removed in 0.28. They are now called `NPE` and `NLE`. The documentation of the stable version is at [sbi.readthedocs.io/en/stable](https://sbi.readthedocs.io/en/stable/).

In [ ]:
import time
import pickle
import tempfile
import torch
import matplotlib.pyplot as plt

from sbi.inference import NPE, NLE, simulate_for_sbi
from sbi.inference.posteriors import MCMCPosteriorParameters
from sbi.neural_nets import posterior_nn
from sbi.utils import BoxUniform, process_prior, process_simulator
from sbi.utils.user_input_checks import check_sbi_inputs
from sbi.analysis import pairplot
from sbi.analysis.plot import sbc_rank_plot, plot_tarp
from sbi.diagnostics import run_sbc, check_sbc, run_tarp, check_tarp

# The toy problem from Notebook 4
from tutorial_utils import (gaussian_simulator, gaussian_benchmark, GAUSSIAN_PRIOR_LOW, GAUSSIAN_PRIOR_HIGH,
                            analytic_moments, plot_posterior_2d, plot_posterior)

# Set seed for reproducibility
torch.manual_seed(42)

# Our models are small, so they do not benefit from many CPU threads. Limiting the thread
# count avoids large slow-downs from thread oversubscription (common on laptops).
torch.set_num_threads(4)

## 1. Prior and Simulations

We use the Gaussian toy model from Notebook 4: the parameters are the mean and the log standard deviation of a Gaussian, the data are ten draws from it.

**Prior.** `sbi` expects the prior as a PyTorch distribution. For a flat prior in a box, use `BoxUniform`.

**Simulations.** `sbi` needs the parameters `theta` with shape `[num_simulations, num_parameters]` and the data `x` with shape `[num_simulations, num_data]`, both as `float32` tensors. Where they come from does not matter: you can run your simulator on a cluster and load the results from disk. We use the same fixed training set as in Notebook 4.

In [ ]:
# Uniform prior over mean [-5, 5] and log_std [-2, 1]
prior = BoxUniform(low=GAUSSIAN_PRIOR_LOW, high=GAUSSIAN_PRIOR_HIGH)

benchmark = gaussian_benchmark()
theta_train, x_train = benchmark.theta_train, benchmark.x_train
theta_test, x_test = benchmark.theta_test, benchmark.x_test
theta_true, x_obs = benchmark.theta_obs, benchmark.x_obs

print("Training parameters:", theta_train.shape, theta_train.dtype)
print("Training data:      ", x_train.shape, x_train.dtype)

If your simulator is a Python function, `sbi` can also run it for you. `process_prior` and `process_simulator` wrap prior and simulator such that they work together, `check_sbi_inputs` tests the combination, and `simulate_for_sbi` draws parameters from the prior and simulates them (optionally in parallel with `num_workers`).

In [ ]:
prior, num_parameters, prior_returns_numpy = process_prior(prior)
simulator = process_simulator(gaussian_simulator, prior, prior_returns_numpy)
check_sbi_inputs(simulator, prior)

theta_demo, x_demo = simulate_for_sbi(simulator, proposal=prior, num_simulations=200, simulation_batch_size=100)
print(theta_demo.shape, x_demo.shape)

## 2. Neural Posterior Estimation (NPE)

NPE trains a neural density estimator (by default a MAF, Masked Autoregressive Flow) to approximate the posterior directly.

This is what we did by hand in Notebook 4: a conditional flow $q(\theta|x)$ for the parameters, with the data as context, trained on the simulated pairs by minimizing $-\frac{1}{N}\sum_i \log q(\theta_i|x_i)$. The `sbi` package takes care of the flow, the normalization of parameters and data, and the training loop (with a validation set and early stopping). The whole analysis is four lines:

In [ ]:
# 1. Choose the method and the density estimator ('maf' is the default)
inference = NPE(prior=prior, density_estimator='maf')

# 2. Pass the simulations
inference = inference.append_simulations(theta_train, x_train)

# 3. Train. Training stops when the validation loss has not improved for 20 epochs (stop_after_epochs).
#    Other useful arguments: training_batch_size=200, learning_rate=5e-4, validation_fraction=0.1, max_num_epochs
density_estimator = inference.train()

# 4. Build the posterior object
posterior = inference.build_posterior(density_estimator)

The training history is stored in `inference.summary`.

*Side effect:* `sbi` writes TensorBoard logs to a folder `sbi-logs` in the current directory.

In [ ]:
print(f"Epochs trained: {inference.summary['epochs_trained'][0]}, best validation loss: {inference.summary['best_validation_loss'][0]:.3f}")

plt.plot(inference.summary['training_loss'], label='Training')
plt.plot(inference.summary['validation_loss'], label='Validation')
plt.xlabel('Epoch')
plt.ylabel(r'$-\log q(\theta|x)$')
plt.title('NPE training')
plt.legend()
plt.show()

### Inference on the Observation

The posterior object can draw samples and evaluate the log-probability for an observation `x`. Note that the number of samples has to be passed as a tuple.

In [ ]:
# Sample from the learned posterior
start = time.time()
samples = posterior.sample((5000,), x=x_obs)
print(f'NPE: {time.time() - start:.2f} s for 5000 posterior samples')

# Log-probability of the true parameters under the learned posterior
print('log q(theta_true | x_obs) =', posterior.log_prob(theta_true.unsqueeze(0), x=x_obs).item())

# sbi's own corner plot for a quick look at posterior samples
fig, axes = pairplot(samples, points=theta_true, labels=['Mean', 'Log Std'], limits=[[-5, 5], [-2, 1]], figsize=(5, 5))

On the scale of the prior, the posterior is a small blob. To see whether it is correct, we zoom in and compare with the exact posterior (with our plotting function from `tutorial_utils.py`).

In [ ]:
plot_posterior({'NPE (sbi)': samples}, x_obs, theta_true)

### Amortization

The network did not learn the posterior for one particular observation. It learned the conditional distribution $q(\theta|x)$ for *any* $x$ which the simulator can produce. Once it is trained, the posterior for a new observation only requires a forward pass through the network: no new simulations and no new training. This is called **amortized** inference, because the cost of the simulations and of the training is paid only once.

`posterior.sample_batched` samples the posteriors of many observations in one call.

In [ ]:
# Three new observations from very different parameters
thetas_new = torch.tensor([[-3.0, 0.0], [0.0, -1.0], [3.0, -0.5]])
x_new = gaussian_simulator(thetas_new)

start = time.time()
samples_new = posterior.sample_batched((5000,), x=x_new, show_progress_bars=False)    # [5000, 3, 2]
print(f'Three posteriors in {time.time() - start:.2f} s, shape of the samples: {tuple(samples_new.shape)}')

fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))
for i, ax in enumerate(axs):
    plot_posterior_2d(ax, samples_new[:, i], x_new[i], thetas_new[i],
                      title=f'True mean = {thetas_new[i, 0]:.1f}, true log std = {thetas_new[i, 1]:.1f}')
axs[0].legend(loc='upper left')
plt.show()

## 3. Neural Likelihood Estimation (NLE)

NLE trains a neural density estimator to approximate the likelihood $p(x|\theta)$: the same kind of conditional flow as for NPE, but with the roles of $\theta$ and $x$ swapped. The posterior then follows from Bayes' theorem, $p(\theta|x) \propto p(x|\theta)\, p(\theta)$, and MCMC is used to obtain posterior samples.

The code is the same as for NPE. The difference is in `build_posterior`: it returns a posterior which runs an MCMC whenever we ask for samples. The sampler is configured with `MCMCPosteriorParameters`. The default method `slice_np_vectorized` is a slice sampler which runs many chains in parallel.

NLE is not amortized: the MCMC has to be run again for every new observation.

In [ ]:
# Instantiate NLE, append the same simulations and train
inference_nle = NLE(prior=prior, density_estimator='maf')
density_estimator_nle = inference_nle.append_simulations(theta_train, x_train).train()

# Build posterior (the MCMC runs when we sample)
mcmc_parameters = MCMCPosteriorParameters(method='slice_np_vectorized', num_chains=50, warmup_steps=100, thin=2)
posterior_nle = inference_nle.build_posterior(density_estimator_nle, posterior_parameters=mcmc_parameters)

# Sample
# MCMC is much slower than sampling from the NPE posterior
start = time.time()
samples_nle = posterior_nle.sample((5000,), x=x_obs)
print(f'NLE: {time.time() - start:.1f} s for 5000 posterior samples')

# Visualize both posteriors
plot_posterior({'NPE (sbi)': samples, 'NLE (sbi)': samples_nle}, x_obs, theta_true)

## 4. Choosing the Density Estimator

The string `'maf'` selects a flow with default settings. Other built-in choices are `'nsf'` (Neural Spline Flow), `'mdn'` (Mixture Density Network) and `'made'`, as well as flows from the `zuko` library (`'zuko_maf'`, `'zuko_nsf'`, ...).

To change the size of the network, use the factory functions `posterior_nn` (for NPE) and `likelihood_nn` (for NLE) and pass the result as `density_estimator`. Their most important arguments:
- `model`: one of the names above
- `hidden_features`, `num_transforms`: width of the networks and number of flow layers (defaults: 50 and 5)
- `z_score_theta`, `z_score_x`: normalization of parameters and data (default: `'independent'`, i.e. every entry separately)
- `embedding_net`: a network which compresses the data before they enter the flow (we use this in Notebook 7)

As an example, we train a Neural Spline Flow, the architecture which we implemented in Notebook 3.

In [ ]:
density_estimator_build_fn = posterior_nn(model='nsf', hidden_features=64, num_transforms=4)

inference_nsf = NPE(prior=prior, density_estimator=density_estimator_build_fn)
posterior_nsf = inference_nsf.build_posterior(inference_nsf.append_simulations(theta_train, x_train).train())

print(f"\nBest validation loss: MAF {inference.summary['best_validation_loss'][0]:.3f}, NSF {inference_nsf.summary['best_validation_loss'][0]:.3f}")

A lower validation loss means a better approximation of the posterior, so the validation loss is the natural way to compare architectures trained on the same simulations.

## 5. How Many Simulations Do We Need?

In real applications, simulations are expensive and their number is the main cost of SBI. What happens if we have too few? We train NPE again with only 500 and 2000 of our simulations and compare with the results for 10000.

For 50 observations from the test set we compare the standard deviation and the mean of the NPE posterior with those of the exact posterior. A width ratio above one means that the NPE posterior is too wide.

In [ ]:
posteriors = {}
for num_simulations in (500, 2000):
    # Train NPE on a subset of the simulations
    inference_small = NPE(prior=prior, density_estimator='maf')
    inference_small = inference_small.append_simulations(theta_train[:num_simulations], x_train[:num_simulations])
    posteriors[f'{num_simulations} simulations (MAF)'] = inference_small.build_posterior(inference_small.train())
posteriors['10000 simulations (MAF)'] = posterior
posteriors['10000 simulations (NSF)'] = posterior_nsf

# Compare with the exact posterior for 50 different observations
x_eval = x_test[:50]
exact = [analytic_moments(x) for x in x_eval]
exact_mean = torch.stack([mean for mean, std in exact])
exact_std = torch.stack([std for mean, std in exact])

fig, axs = plt.subplots(1, 4, figsize=(20, 4.5))
print('\n                        | width of NPE posterior / width of exact posterior | |bias| / width of exact posterior')
print('                        |          Mean      Log Std                        |      Mean      Log Std')
for ax, (name, post) in zip(axs, posteriors.items()):
    s = post.sample_batched((2000,), x=x_eval, show_progress_bars=False)          # [2000, 50, 2]
    # Median over the 50 observations
    width_ratio = (s.std(dim=0) / exact_std).median(dim=0).values
    bias = ((s.mean(dim=0) - exact_mean).abs() / exact_std).median(dim=0).values
    print(f'{name:23s} | {width_ratio[0]:13.2f} {width_ratio[1]:12.2f}                        | {bias[0]:9.2f} {bias[1]:12.2f}')

    # The posterior for our observation x_obs
    plot_posterior_2d(ax, post.sample((5000,), x=x_obs, show_progress_bars=False), x_obs, theta_true, title=name)
axs[0].legend(loc='upper left')
plt.show()

With only 500 or 2000 simulations, the network has not seen enough examples to learn the posterior precisely: the posteriors (in particular for the log standard deviation) are much too wide and their centre is off. The results for small numbers of simulations also change considerably from one training run to the next. With 10000 simulations we are close to the exact posterior, and the spline flow, which had the lower validation loss, is closer than the MAF.

Usually there is no exact posterior to compare with. A practical check is then to increase the number of simulations until the posteriors stop changing, together with the coverage tests of the next section.

## 6. Diagnostics: Coverage Tests

It is crucial to verify that the inferred posteriors are well-calibrated: if a region contains $x\%$ of the posterior probability mass, it should contain the true parameters $x\%$ of the time. `sbi` provides several such tests in `sbi.diagnostics`. All of them take test simulations `(theta_test, x_test)` and the posterior object.

### TARP

This is the test we implemented in Notebook 4. `run_tarp` returns the expected coverage `ecp` for the credibility levels `alpha`. `check_tarp` summarizes the curve in two numbers: the area between the curve and the diagonal (`atc`, should be close to zero) and the p-value of a Kolmogorov-Smirnov test (should not be small).

In [ ]:
num_test = 300
ecp, alpha = run_tarp(theta_test[:num_test], x_test[:num_test], posterior, num_posterior_samples=1000,
                      use_batched_sampling=False, show_progress_bar=False)
atc, ks_pvalue = check_tarp(ecp, alpha)
print(f'TARP: area to the diagonal = {atc:.3f}, KS p-value = {ks_pvalue:.2f}')

fig, ax = plot_tarp(ecp, alpha)

### Simulation-Based Calibration (SBC)

TARP tests all parameters together. SBC looks at each parameter separately: for every test simulation it computes the *rank* of the true parameter among the posterior samples. If the posteriors are correct, these ranks are uniformly distributed.
- A histogram which is peaked in the middle means that the posteriors are too wide.
- A U-shaped histogram means that they are too narrow.
- A tilted histogram means that they are biased.

`check_sbc` returns p-values of a test for uniformity (`ks_pvals`, one per parameter).

In [ ]:
ranks, dap_samples = run_sbc(theta_test[:num_test], x_test[:num_test], posterior, num_posterior_samples=1000,
                             use_batched_sampling=False, show_progress_bar=False)
check_stats = check_sbc(ranks, theta_test[:num_test], dap_samples, num_posterior_samples=1000)
print('SBC: KS p-values for (Mean, Log Std):', check_stats['ks_pvals'].numpy().round(3))

# The grey band shows the range expected for a uniform distribution
fig, ax = sbc_rank_plot(ranks, 1000, plot_type='hist', num_bins=15, parameter_labels=['Mean', 'Log Std'])

### How to Read a Coverage Plot

To learn what a failed coverage test looks like, we break our posteriors on purpose. We take the NPE samples of every test simulation and
- shrink them around their mean by a factor of two (posterior too narrow, *overconfident*),
- stretch them by a factor of two (posterior too wide, *underconfident*),
- shift them by one standard deviation (*biased*).

Since we now work with arrays of samples and not with a posterior object, we use the standalone [`tarp`](https://github.com/Ciela-Institute/tarp) package, as in Notebook 4.

What to look for in the plot below:
- **Too narrow:** the curve is flatter than the diagonal. It is above the diagonal at low credibility levels and below it at high ones: the true parameters are in the tails of the posterior too often.
- **Too wide:** the opposite, the curve is steeper than the diagonal in the middle.
- **Biased:** looks similar to a posterior which is too narrow. The coverage test tells us *that* something is wrong, but not always *what*.

In [ ]:
from tarp import get_tarp_coverage

# Posterior samples for all test simulations: [num_samples, num_test, num_params]
posterior_samples = posterior.sample_batched((1000,), x=x_test, show_progress_bars=False)

def distort(samples, scale=1.0, shift=0.0):
    # Rescale the samples of every test case around their mean and shift them (in units of their standard deviation)
    mean = samples.mean(dim=0, keepdim=True)
    std = samples.std(dim=0, keepdim=True)
    return mean + scale * (samples - mean) + shift * std

variants = {
    'NPE posterior': posterior_samples,
    'Too narrow (overconfident)': distort(posterior_samples, scale=0.5),
    'Too wide (underconfident)': distort(posterior_samples, scale=2.0),
    'Biased': distort(posterior_samples, shift=1.0),
}

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], 'k--', label='Ideal')
for name, variant in variants.items():
    # norm=True rescales the parameters to the unit hypercube, from which the random reference points are drawn
    ecp, alpha = get_tarp_coverage(variant.numpy(), theta_test.numpy(), references='random', metric='euclidean', norm=True)
    plt.plot(alpha, ecp, label=name)
plt.xlabel('Credibility Level')
plt.ylabel('Expected Coverage')
plt.title('TARP for deliberately wrong posteriors')
plt.legend()
plt.show()

## 7. Saving and Loading

Posterior objects (and inference objects) can be saved with `pickle`.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    with open(f'{directory}/posterior.pkl', 'wb') as handle:
        pickle.dump(posterior, handle)
    with open(f'{directory}/posterior.pkl', 'rb') as handle:
        posterior_loaded = pickle.load(handle)

print('Samples from the loaded posterior:', posterior_loaded.sample((3,), x=x_obs, show_progress_bars=False).numpy().round(2).tolist())

## Cheat Sheet and Where to Go Next

| Step | Code |
|---|---|
| Prior | `prior = BoxUniform(low, high)` (or any PyTorch distribution) |
| Simulations | `theta` `[N, num_parameters]`, `x` `[N, num_data]`, `float32` tensors |
| Train NPE | `inference = NPE(prior, density_estimator='maf')`<br>`inference.append_simulations(theta, x).train()` |
| Train NLE | the same with `NLE` |
| Posterior | `posterior = inference.build_posterior()` |
| Samples | `posterior.sample((n,), x=x_obs)`, many observations: `posterior.sample_batched((n,), x=x_batch)` |
| Density | `posterior.log_prob(theta, x=x_obs)` |
| Custom network | `density_estimator=posterior_nn(model='nsf', hidden_features=64, num_transforms=4, embedding_net=...)` |
| MCMC settings | `build_posterior(posterior_parameters=MCMCPosteriorParameters(num_chains=50, warmup_steps=100))` |
| Diagnostics | `run_tarp`, `check_tarp`, `run_sbc`, `check_sbc` in `sbi.diagnostics`, plots in `sbi.analysis.plot` |

Things we did not cover (see the [documentation](https://sbi.readthedocs.io/en/stable/)):
- **Sequential (multi-round) inference:** if you only care about one observation, you can simulate where it matters. After each round, the current posterior is used as the proposal for the next simulations: `inference.append_simulations(theta, x, proposal=proposal).train()` with `proposal = posterior.set_default_x(x_obs)`. The result is no longer amortized.
- **Other methods:** neural ratio estimation (`NRE`), flow matching (`FMPE`) and score-based diffusion models (`NPSE`) follow the same four-line pattern.
- **Other samplers for NLE:** variational inference (`sample_with='vi'`) instead of MCMC.

## Summary

- With `sbi`, a complete NPE or NLE analysis takes four lines. The package handles the flow, the normalization, the training loop and the sampling.
- **NPE** is *amortized*: after training, the posterior for any new observation costs a fraction of a second.
- **NLE** needs MCMC for every observation. In return, the prior only enters at sampling time (we will make use of this in Notebook 7).
- The quality of the result depends on the number of simulations. With too few, the posteriors are typically too wide and can be biased.
- In a real application there is no exact posterior to compare with. Coverage tests like TARP and SBC are then the most important check.

**Next:** Notebook 6 solves the same problem with `ltu-ili`, a pipeline built on top of `sbi` which is widely used in cosmology.